# Model 2 — Predictive maintenance: data audit, leakage and feasibility (Sprint 3)

**Goal.** Decide whether the datasets in the research PDFs can honestly support an XGBoost model that outputs a *failure probability* for a vehicle
(and, separately, for **engine / brakes / electrical**), using only information the live system has.

**Scope: experiments only.** No final training, no `.joblib`, no backend integration; the old `.joblib` files are not used.

**What the live system really has for an arbitrary plate**
* **Registry (data.gov.il):** make, model text, manufacture year (→ age), fuel, ownership category. *No* mileage, *no* sensors.
* **Model 1:** an *estimate* of the odometer (uncertain: see notebook 01).
* **PostgreSQL `maintenance_records`:** history **only for vehicles that already have records** — for a random plate typed into the search box it is empty.
* **No** IoT/OBD/CAN data, no engine temperature, no oil quality, no brake sensors, no route/weather/load.

**Method.** (1) audit each file; (2) classify every column (available at runtime? leakage?); (3) *reverse-engineer how the label was produced*;
(4) fit honest models under the realistic availability scenarios and under an "all columns" upper bound; (5) test component-level feasibility;
(6) look for real, official component-level information; (7) state what is technically defensible.

In [1]:
import os, sys, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.options.display.width = 170
pd.options.display.max_columns = 60
pd.options.display.max_rows = 100
pd.options.display.float_format = "{:,.3f}".format
sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import RAW_DIR, INTERIM_DIR
from israel_registry import normalise_registry
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, f1_score, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
try:
    display
except NameError:
    display = print
import sklearn, xgboost, imblearn
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__, "| imblearn", imblearn.__version__)

python 3.13.13 | sklearn 1.9.0 | xgboost 3.4.1 | imblearn 0.14.2


## 1. Data and provenance

In [2]:
D1 = RAW_DIR / "d1_logistics"
v2 = pd.read_csv(D1 / "logistics_predictive_maintenanceV2.csv")          # 250k rows, 50 cols (richest version)
d92 = pd.read_csv(D1 / "logistics_dataset_with_maintenance_required.csv")  # 92k rows, 27 cols
rev = pd.read_csv(D1 / "revised_logistics_dataset_V2.csv")                  # 250k rows, 30 cols
d4 = pd.read_csv(RAW_DIR / "d4_telemetry" / "synthetic_telemetry_data.csv")
d5 = pd.read_csv(RAW_DIR / "d5_nev_faults" / "Fault_nev_dataset.csv")

def sha256(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()[:16]
files = sorted(D1.glob("*.csv")) + [RAW_DIR / "d4_telemetry" / "synthetic_telemetry_data.csv", RAW_DIR / "d5_nev_faults" / "Fault_nev_dataset.csv"]
display(pd.DataFrame([{"file": str(p.relative_to(RAW_DIR)), "MB": p.stat().st_size / 1e6, "sha256[:16]": sha256(p)} for p in files]))

d4["any_fault"] = (d4[["engine_failure_imminent", "brake_issue_imminent", "battery_issue_imminent"]].sum(axis=1) > 0).astype(int)
d5["is_fault"] = (d5.fault_type != "no_fault").astype(int)

def audit(name, df, target=None, idcol=None):
    cols = [c for c in df.columns if c != idcol]
    return {"dataset": name, "rows": len(df), "cols": df.shape[1], "null cells": int(df.isna().sum().sum()),
            "exact duplicate rows (excl. id)": int(df[cols].duplicated().sum()), "unique ids": (df[idcol].nunique() if idcol else np.nan),
            "target": target, "prevalence": (df[target].mean() if target else np.nan)}
display(pd.DataFrame([
    audit("D1 V2 (250k)", v2, "Maintenance_Required", "Vehicle_ID"),
    audit("D1 main (92k)", d92, "Maintenance_Required", "Vehicle_ID"),
    audit("D1 revised V2 (250k)", rev, "Maintenance_Required", "Vehicle_ID"),
    audit("D4 telemetry (50 vehicles)", d4, "any_fault", "vehicle_id"),
    audit("D5 NEV faults", d5, "is_fault"),
]).set_index("dataset"))
print("Maintenance_Severity (V2):", v2.Maintenance_Severity.value_counts(normalize=True).round(3).to_dict())

,file,MB,sha256[:16]
0,d1_logistics\logistics_dataset_with_maintenanc...,25.068,e27eeb21ebac0903
1,d1_logistics\logistics_predictive_maintenanceV...,98.359,279cafcaa5cc077d
2,d1_logistics\revised_logistics_dataset_V2.csv,95.332,f18834ad0dbd052c
3,d4_telemetry\synthetic_telemetry_data.csv,1.292,f7dd2c8f7941ccc4
4,d5_nev_faults\Fault_nev_dataset.csv,0.257,bec888a27f31e688


,rows,cols,null cells,exact duplicate rows (excl. id),unique ids,target,prevalence
dataset,,,,,,,
D1 V2 (250k),250000,50,0,0,"250,000.000",Maintenance_Required,0.300
D1 main (92k),92000,27,0,0,"92,000.000",Maintenance_Required,0.768
D1 revised V2 (250k),250000,30,0,0,"250,000.000",Maintenance_Required,0.768
D4 telemetry (50 vehicles),1970,42,0,0,50.000,any_fault,0.018
D5 NEV faults,1000,19,0,0,NaN,is_fault,0.795


Maintenance_Severity (V2): {'Normal': 0.7, 'Minor Maintenance': 0.2, 'Major Maintenance': 0.1}


**First observations.** No nulls and no duplicates anywhere, and every `Vehicle_ID` is unique — the usual red flags are absent. But the three Dataset-1 files have
**different prevalence (30% vs 77%)**, i.e. they were generated by *different labelling rules*. That already says the label is a construction, not a measured outcome.

## 2. Column-by-column audit: runtime availability and leakage

In [3]:
def single_col_auc(df, target):
    y = df[target]; rows = []
    for c in df.columns:
        if c in (target, "Vehicle_ID"):
            continue
        s = df[c]
        if not pd.api.types.is_numeric_dtype(s):
            if s.nunique() > 50:
                continue
            s = df[c].map(df.groupby(c)[target].mean())   # in-sample target mean (optimistic for sparse categories)
        a = roc_auc_score(y, s.fillna(s.median()))
        rows.append({"column": c, "AUC": max(a, 1 - a),
                     "constant when y=0": df.loc[y == 0, c].nunique() == 1, "constant when y=1": df.loc[y == 1, c].nunique() == 1})
    return pd.DataFrame(rows).set_index("column")

auc_v2 = single_col_auc(v2, "Maintenance_Required")
auc_92 = single_col_auc(d92, "Maintenance_Required")

AV = {  # column -> (what the live system could provide, verdict)
 "Vehicle_ID": ("-", "not a feature (identifier)"),
 "Make_and_Model": ("registry make/model", "usable in principle, but only 4-8 heavy-truck models: no match with Israeli passenger cars"),
 "Year_of_Manufacture": ("registry shnat_yitzur -> age", "USABLE"),
 "Vehicle_Type": ("registry only has P/M, not this taxonomy", "partial / no equivalent"),
 "Usage_Hours": ("telematics (closest runtime analogue = Model-1 km estimate, a different quantity)", "NOT available"),
 "Route_Info": ("operational telematics", "NOT available"),
 "Weather_Conditions": ("operational telematics", "NOT available"),
 "Road_Conditions": ("operational telematics", "NOT available"),
 "Delivery_Times": ("operational telematics", "NOT available"),
 "Load_Capacity": ("vehicle spec (not in registry)", "NOT available (spec lookup at best)"),
 "Actual_Load": ("operational telematics", "NOT available"),
 "Last_Maintenance_Date": ("PostgreSQL maintenance_records", "available ONLY if the plate has records"),
 "Days_Since_Last_Maintenance": ("PostgreSQL maintenance_records", "available ONLY if the plate has records; in V2 it DEFINES the label"),
 "Maintenance_Type": ("PostgreSQL maintenance_records (category of last record)", "available ONLY if records exist; in V2 it is fully determined by the label (LEAK)"),
 "Maintenance_Cost": ("PostgreSQL maintenance_records", "available ONLY if records exist; V2: derived from label (LEAK)"),
 "Historical_Maintenance_Cost": ("PostgreSQL maintenance_records", "available ONLY if records exist; V2: derived from label (LEAK)"),
 "Failure_History": ("PostgreSQL (count of past failure events)", "available ONLY if records exist; DEFINES the label in the 92k file"),
 "Engine_Temperature": ("OBD/IoT sensor", "NOT available"),
 "Tire_Pressure": ("IoT sensor", "NOT available"),
 "Fuel_Consumption": ("OBD/telematics", "NOT available"),
 "Battery_Status": ("IoT sensor", "NOT available"),
 "Vibration_Levels": ("IoT sensor", "NOT available"),
 "Oil_Quality": ("IoT sensor", "NOT available; DEFINES the label in the 92k file"),
 "Brake_Condition": ("sensor / inspection", "NOT available; DEFINES the label in the 92k file"),
 "Anomalies_Detected": ("diagnostic monitoring", "NOT available; DEFINES the label in the 92k file"),
 "Diagnostic_Trouble_Code_Count": ("OBD", "NOT available"),
 "CAN_Message_Rate_Hz": ("CAN bus", "NOT available"),
 "Sensor_Packet_Loss_Rate": ("telematics link quality", "NOT available"),
 "Downtime_Maintenance": ("post-event outcome", "LEAK (consequence of the maintenance event)"),
 "Impact_on_Efficiency": ("post-event outcome", "LEAK (consequence of the maintenance event)"),
 "Predictive_Score": ("pre-computed score of unknown origin", "NOT available; V2: AUC 0.9999 (LEAK); 92k: AUC 0.50 (noise)"),
 "Maintenance_Severity": ("alternative encoding of the target", "LEAK"),
 "Maintenance_Severity_ID": ("alternative encoding of the target", "LEAK"),
 "Maintenance_Level": ("alternative encoding of the target", "LEAK"),
 "Maintenance_Level_Code": ("alternative encoding of the target", "LEAK"),
 "Severity_Score": ("alternative encoding of the target", "LEAK"),
 "PCR": ("derived from load fields", "NOT available"), "UIR": ("derived from hours/age", "NOT available"),
 "TPI": ("derived from sensors", "NOT available"), "MBF": ("derived from cost/failure history", "LEAK-prone (V2 AUC 0.99)"),
 "ADS": ("derived from anomalies/DTC", "NOT available / LEAK-prone"), "OHI": ("derived from sensors", "NOT available"),
 "CMES": ("derived from consumption/downtime", "LEAK-prone (V2 AUC 0.999)"), "UER": ("derived from hours/cost", "LEAK-prone"),
}
for c in ["Communication_Interface", "Telematics_Gateway", "Edge_Device_Class", "Pre_Event_Record", "Data_Split", "FL_Client_ID", "Partition_Type",
          "Local_Epochs_Per_Round", "Communication_Rounds"]:
    AV[c] = ("experiment / telematics metadata", "not a feature")
cols_all = sorted(set(v2.columns) | set(d92.columns) - {"Vehicle_ID"})
table = pd.DataFrame({"AUC in V2": auc_v2["AUC"], "AUC in 92k": auc_92["AUC"]}).reindex([c for c in cols_all if c not in ("Maintenance_Required", "Vehicle_ID")])
table["runtime source"] = [AV.get(c, ("?", "?"))[0] for c in table.index]
table["verdict"] = [AV.get(c, ("?", "?"))[1] for c in table.index]
display(table.sort_values("AUC in V2", ascending=False))

,AUC in V2,AUC in 92k,runtime source,verdict
column,,,,
Days_Since_Last_Maintenance,1.000,NaN,PostgreSQL maintenance_records,available ONLY if the plate has records; in V2...
Maintenance_Type,1.000,0.503,PostgreSQL maintenance_records (category of la...,available ONLY if records exist; in V2 it is f...
Maintenance_Severity_ID,1.000,NaN,alternative encoding of the target,LEAK
Maintenance_Severity,1.000,NaN,alternative encoding of the target,LEAK
Predictive_Score,1.000,0.502,pre-computed score of unknown origin,NOT available; V2: AUC 0.9999 (LEAK); 92k: AUC...
CMES,0.999,NaN,derived from consumption/downtime,LEAK-prone (V2 AUC 0.999)
MBF,0.994,NaN,derived from cost/failure history,LEAK-prone (V2 AUC 0.99)
Maintenance_Cost,0.990,0.503,PostgreSQL maintenance_records,available ONLY if records exist; V2: derived f...
Historical_Maintenance_Cost,0.990,NaN,PostgreSQL maintenance_records,available ONLY if records exist; V2: derived f...


Columns with AUC ≈ 1 on their own are not "good predictors": they are **the label or a copy of it**. A real maintenance outcome is never separated perfectly by one column.

## 3. Reverse-engineering the label
If a label can be *reproduced exactly by a rule over other columns*, it is not an observed failure — it is a synthetic labelling rule.

In [4]:
rule_v2 = (v2.Days_Since_Last_Maintenance > 180).astype(int)
print(f"V2: Maintenance_Required == (Days_Since_Last_Maintenance > 180) in {(rule_v2 == v2.Maintenance_Required).mean():.4%} of {len(v2):,} rows")
print(v2.groupby("Maintenance_Required").Days_Since_Last_Maintenance.agg(["min", "max"]))
print()
display(pd.crosstab(v2.Maintenance_Type, v2.Maintenance_Required))
display(pd.crosstab(v2.Maintenance_Severity, v2.Maintenance_Required))

V2: Maintenance_Required == (Days_Since_Last_Maintenance > 180) in 100.0000% of 250,000 rows
                      min  max
Maintenance_Required          
0                      30  180
1                     181  730



Maintenance_Required,0,1
Maintenance_Type,,
Battery Service,0,12551
Brake Service,0,12510
Cooling System Repair,0,6164
Electrical Repair,0,6321
Engine Overhaul,0,6253
Oil Change,0,12459
Oil Check,58377,0
Routine Inspection,58363,0
Tire Replacement,0,12480


Maintenance_Required,0,1
Maintenance_Severity,,
Major Maintenance,0,25000
Minor Maintenance,0,50000
Normal,175000,0


In [5]:
y92 = d92.Maintenance_Required.astype(bool)
r_or = (d92.Anomalies_Detected > 0) | (d92.Failure_History > 0) | (d92.Brake_Condition == "Poor")
leftover = d92[~r_or]
r_full = r_or | (d92.Oil_Quality < 50.3)
print(f"92k: (Anomalies>0) OR (Failure_History>0) OR (Brake_Condition=='Poor') reproduces {(r_or == y92).mean():.4%} of rows (false positives: {int((r_or & ~y92).sum())})")
print(f"     adding OR (Oil_Quality < 50.3) -> {(r_full == y92).mean():.4%} exact match of {len(d92):,} rows")
print("Remaining positives that only the Oil_Quality clause explains:", int(leftover.Maintenance_Required.sum()))
display(pd.crosstab(d92.Brake_Condition, d92.Maintenance_Required))
display(pd.crosstab(d92.Failure_History, d92.Maintenance_Required))
display(pd.crosstab(d92.Anomalies_Detected, d92.Maintenance_Required))

92k: (Anomalies>0) OR (Failure_History>0) OR (Brake_Condition=='Poor') reproduces 99.9761% of rows (false positives: 0)
     adding OR (Oil_Quality < 50.3) -> 100.0000% exact match of 92,000 rows
Remaining positives that only the Oil_Quality clause explains: 22


Maintenance_Required,0,1
Brake_Condition,,
Fair,6116,12333
Good,15229,30709
Poor,0,27613


Maintenance_Required,0,1
Failure_History,,
0,21345,33895
1,0,36760


Maintenance_Required,0,1
Anomalies_Detected,,
0,21345,29053
1,0,41602


**Conclusion of the audit.**
* **V2:** `Maintenance_Required ≡ Days_Since_Last_Maintenance > 180`; `Maintenance_Type` and `Maintenance_Severity` are deterministic given the label.
  A model trained on it learns a *calendar rule*, not failure physics.
* **92k:** `Maintenance_Required ≡ Anomalies_Detected>0 OR Failure_History>0 OR Brake_Condition=Poor OR Oil_Quality<50.3` — a 4-clause OR.
  Three clauses are sensor/diagnostic readings that do not exist at runtime; **`Failure_History` is a defining clause too, so it is leakage-adjacent**
  (my first analysis, which treated it as a legitimate predictor, was wrong).
* Neither file contains an *observed* failure outcome. Neither contains a mileage column (only `Usage_Hours`).

## 4. Honest models under realistic availability scenarios
Scenarios (features):
* **G — registry only**: vehicle age, make/model, vehicle type. *This is what exists for a random plate.*
* **H1 — G + days since last maintenance** (needs a PostgreSQL record). **H2 — G + Failure_History** (needs recorded failures). **H — both.**
* **U — upper bound, ALL columns** (sensors, derived scores…): impossible at runtime; only shows whether the label is learnable at all.

Protocol: stratified 60/20/20 train/validation/test (rows are unique vehicles, so no group leakage). The decision threshold is chosen on **validation**
(highest precision with recall ≥ 0.90) and applied once to **test**. Baselines: *always predict "at risk"* (recall 1.0 by construction) and the prior.
Imbalance handling compared: LogisticRegression with `class_weight`, XGBoost with `scale_pos_weight`, XGBoost + SMOTE (the PDF's two pipelines).

In [6]:
def make_prep(cat, num, scale):
    tr = []
    if num: tr.append(("num", StandardScaler() if scale else "passthrough", num))
    if cat: tr.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat))
    return ColumnTransformer(tr)

def xgb(**kw):
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, tree_method="hist", n_jobs=8, random_state=0, **kw)

def candidate_models(pos_weight, cat, num, with_smote=True):
    m = {
        "LogReg (class_weight=balanced)": Pipeline([("p", make_prep(cat, num, True)), ("m", LogisticRegression(max_iter=3000, class_weight="balanced"))]),
        "XGBoost (scale_pos_weight)": Pipeline([("p", make_prep(cat, num, False)), ("m", xgb(scale_pos_weight=pos_weight))]),
    }
    if with_smote:
        m["XGBoost + SMOTE"] = ImbPipeline([("p", make_prep(cat, num, False)), ("s", SMOTE(random_state=0)), ("m", xgb())])
    return m

def threshold_for_recall(y, p, target=0.90):
    best = None
    for t in np.quantile(p, np.linspace(0, 1, 401)):
        pred = p >= t
        if recall_score(y, pred) >= target:
            pr = precision_score(y, pred, zero_division=0)
            if best is None or pr > best[1]:
                best = (t, pr)
    return best[0] if best else 0.5

def boot_auc_ci(y, p, n=200, seed=0):
    rng = np.random.default_rng(seed); y = np.asarray(y); p = np.asarray(p); out = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i):
            out.append(roc_auc_score(y[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

def run(df, target, scenarios, label, smote_in=("G", "H1", "H2", "H")):
    y = df[target].astype(int)
    tr, tmp = train_test_split(df.index, test_size=0.4, stratify=y, random_state=0)
    va, te = train_test_split(tmp, test_size=0.5, stratify=y[tmp], random_state=0)
    prev = y[te].mean(); rows = []; probs = {}
    rows.append({"dataset": label, "scenario": "-", "model": "Baseline: always predict at-risk", "ROC-AUC": 0.5, "PR-AUC": prev, "lift": 1.0,
                 "Recall": 1.0, "Precision": prev, "F1": 2 * prev / (1 + prev), "Brier": np.nan, "AUC 95% CI": "-"})
    for sname, (cat, num) in scenarios.items():
        X = df[cat + num]
        pw = (y[tr] == 0).sum() / (y[tr] == 1).sum()
        for mname, model in candidate_models(pw, cat, num, with_smote=sname in smote_in).items():
            model.fit(X.loc[tr], y[tr])
            pv, pt = model.predict_proba(X.loc[va])[:, 1], model.predict_proba(X.loc[te])[:, 1]
            thr = threshold_for_recall(y[va], pv); pred = pt >= thr
            ci = boot_auc_ci(y[te], pt)
            rows.append({"dataset": label, "scenario": sname, "model": mname, "ROC-AUC": roc_auc_score(y[te], pt), "PR-AUC": average_precision_score(y[te], pt),
                         "lift": average_precision_score(y[te], pt) / prev, "Recall": recall_score(y[te], pred), "Precision": precision_score(y[te], pred, zero_division=0),
                         "F1": f1_score(y[te], pred), "Brier": brier_score_loss(y[te], pt), "AUC 95% CI": f"[{ci[0]:.3f}, {ci[1]:.3f}]"})
            probs[(sname, mname)] = (np.asarray(y[te]), pt)
    return pd.DataFrame(rows), probs

# ---- feature engineering for both files
v2 = v2.assign(age=2025 - v2.Year_of_Manufacture)          # V2: implied reference date is 2025-01-01
d92 = d92.assign(age=2024 - d92.Year_of_Manufacture)
d92["Days_Since_Last_Maintenance"] = (pd.to_datetime(d92.Last_Maintenance_Date).max() + pd.Timedelta(days=1) - pd.to_datetime(d92.Last_Maintenance_Date)).dt.days

G_cat, G_num = ["Make_and_Model", "Vehicle_Type"], ["age"]
SC = {
    "G": (G_cat, G_num),
    "H1": (G_cat, G_num + ["Days_Since_Last_Maintenance"]),
    "H2": (G_cat, G_num + ["Failure_History"]),
    "H": (G_cat, G_num + ["Days_Since_Last_Maintenance", "Failure_History"]),
}
def upper(df, drop):
    cols = [c for c in df.columns if c not in drop]
    cat = [c for c in cols if not pd.api.types.is_numeric_dtype(df[c])]
    return cat, [c for c in cols if c not in cat]
drop_v2 = ["Vehicle_ID", "Maintenance_Required", "Maintenance_Severity", "Maintenance_Severity_ID", "Last_Maintenance_Date", "Data_Split", "Pre_Event_Record"]
drop_92 = ["Vehicle_ID", "Maintenance_Required", "Last_Maintenance_Date"]
res_v2, probs_v2 = run(v2, "Maintenance_Required", {**SC, "U (ALL columns)": upper(v2, drop_v2)}, "V2 (prevalence 30%)")
display(res_v2)

,dataset,scenario,model,ROC-AUC,PR-AUC,lift,Recall,Precision,F1,Brier,AUC 95% CI
0,V2 (prevalence 30%),-,Baseline: always predict at-risk,0.500,0.300,1.000,1.000,0.300,0.462,NaN,-
1,V2 (prevalence 30%),G,LogReg (class_weight=balanced),0.504,0.303,1.010,0.912,0.300,0.452,0.250,"[0.499, 0.509]"
2,V2 (prevalence 30%),G,XGBoost (scale_pos_weight),0.505,0.304,1.013,0.909,0.300,0.451,0.250,"[0.500, 0.511]"
3,V2 (prevalence 30%),G,XGBoost + SMOTE,0.505,0.303,1.010,0.935,0.300,0.455,0.250,"[0.499, 0.511]"
4,V2 (prevalence 30%),H1,LogReg (class_weight=balanced),1.000,1.000,3.333,0.991,1.000,0.995,0.003,"[1.000, 1.000]"
5,V2 (prevalence 30%),H1,XGBoost (scale_pos_weight),1.000,1.000,3.333,1.000,1.000,1.000,0.000,"[1.000, 1.000]"
6,V2 (prevalence 30%),H1,XGBoost + SMOTE,1.000,1.000,3.333,1.000,1.000,1.000,0.000,"[1.000, 1.000]"
7,V2 (prevalence 30%),H2,LogReg (class_weight=balanced),0.838,0.791,2.636,0.904,0.371,0.526,0.133,"[0.835, 0.842]"
8,V2 (prevalence 30%),H2,XGBoost (scale_pos_weight),0.837,0.790,2.632,0.897,0.374,0.528,0.133,"[0.833, 0.840]"
9,V2 (prevalence 30%),H2,XGBoost + SMOTE,0.838,0.791,2.636,0.900,0.375,0.530,0.133,"[0.834, 0.842]"


In [7]:
res_92, probs_92 = run(d92, "Maintenance_Required", {**SC, "U (ALL columns)": upper(d92, drop_92)}, "92k (prevalence 77%)")
display(res_92)

,dataset,scenario,model,ROC-AUC,PR-AUC,lift,Recall,Precision,F1,Brier,AUC 95% CI
0,92k (prevalence 77%),-,Baseline: always predict at-risk,0.500,0.768,1.000,1.000,0.768,0.869,NaN,-
1,92k (prevalence 77%),G,LogReg (class_weight=balanced),0.501,0.769,1.001,0.924,0.767,0.838,0.250,"[0.490, 0.510]"
2,92k (prevalence 77%),G,XGBoost (scale_pos_weight),0.508,0.774,1.007,0.966,0.769,0.856,0.250,"[0.499, 0.518]"
3,92k (prevalence 77%),G,XGBoost + SMOTE,0.504,0.771,1.004,0.901,0.770,0.830,0.250,"[0.494, 0.514]"
4,92k (prevalence 77%),H1,LogReg (class_weight=balanced),0.505,0.772,1.005,0.905,0.767,0.830,0.250,"[0.496, 0.514]"
5,92k (prevalence 77%),H1,XGBoost (scale_pos_weight),0.502,0.772,1.006,1.000,0.768,0.869,0.250,"[0.494, 0.512]"
6,92k (prevalence 77%),H1,XGBoost + SMOTE,0.499,0.769,1.001,0.917,0.767,0.835,0.193,"[0.488, 0.508]"
7,92k (prevalence 77%),H2,LogReg (class_weight=balanced),0.766,0.929,1.209,0.900,0.792,0.842,0.192,"[0.760, 0.772]"
8,92k (prevalence 77%),H2,XGBoost (scale_pos_weight),0.765,0.928,1.209,0.913,0.791,0.848,0.192,"[0.759, 0.772]"
9,92k (prevalence 77%),H2,XGBoost + SMOTE,0.764,0.928,1.208,0.905,0.791,0.844,0.192,"[0.758, 0.771]"


**How to read the two tables**
* **Scenario G (what exists for a random plate): ROC-AUC 0.50–0.51 (CI within ±0.01 of 0.5), lift ≤ 1.013.** Age, make and vehicle type carry *no* usable information about the label,
  in either file, with any algorithm or imbalance strategy. Recall at the chosen threshold is high only because the threshold becomes "flag almost everything"
  (compare with the *always at-risk* baseline: same recall/precision/F1).
* **Scenarios with `Days_Since_Last_Maintenance` (V2) or `Failure_History` (92k)** reach high AUC — but that is the model *recovering the labelling rule* shown in section 3, not predicting failures.
* "Optimising Recall" is **meaningless on the 92k file**: with 77% positives, flagging every vehicle gives Recall 1.0, Precision 0.77, F1 0.87. The PDF's premise (failure = rare minority event, hence SMOTE) does not hold for this file;
  it holds for V2 (30%) but there the label is a calendar rule.

In [8]:
def calibration(y, p, bins=8):
    d = pd.DataFrame({"y": y, "p": p}); d["bin"] = pd.qcut(d.p, bins, duplicates="drop")
    t = d.groupby("bin", observed=True).agg(mean_predicted=("p", "mean"), observed_rate=("y", "mean"), n=("y", "size"))
    t["abs_gap"] = (t.mean_predicted - t.observed_rate).abs()
    return t, float((t.abs_gap * t.n).sum() / t.n.sum())
for lab, probs, sc in [("92k / scenario H2 (Failure_History)", probs_92, "H2"), ("92k / scenario G", probs_92, "G")]:
    y_, p_ = probs[(sc, "XGBoost (scale_pos_weight)")]
    t, ece = calibration(y_, p_)
    print(f"{lab} - XGBoost scale_pos_weight: expected calibration error = {ece:.3f}  (scale_pos_weight deliberately distorts probabilities)")
    display(t)

92k / scenario H2 (Failure_History) - XGBoost scale_pos_weight: expected calibration error = 0.172  (scale_pos_weight deliberately distorts probabilities)


,mean_predicted,observed_rate,n,abs_gap
bin,,,,
"(0.19885, 0.31018]",0.300,0.597,2682,0.297
"(0.31018, 0.31733]",0.314,0.619,2153,0.305
"(0.31733, 0.33195]",0.325,0.624,2439,0.298
"(0.33195, 0.3414]",0.338,0.598,1977,0.261
"(0.3414, 0.9998]",0.501,0.709,2251,0.208
"(0.9998, 0.99981]",1.000,1.000,2405,0.000
"(0.99981, 0.99982]",1.000,1.000,2527,0.000
"(0.99982, 0.99989]",1.000,1.000,1966,0.000


92k / scenario G - XGBoost scale_pos_weight: expected calibration error = 0.268  (scale_pos_weight deliberately distorts probabilities)


,mean_predicted,observed_rate,n,abs_gap
bin,,,,
"(0.356, 0.475]",0.460,0.749,2331,0.289
"(0.475, 0.489]",0.485,0.767,2505,0.282
"(0.489, 0.494]",0.492,0.782,2340,0.290
"(0.494, 0.497]",0.496,0.769,2076,0.273
"(0.497, 0.506]",0.501,0.773,2251,0.271
"(0.506, 0.516]",0.513,0.749,2350,0.236
"(0.516, 0.522]",0.518,0.772,2281,0.253
"(0.522, 0.684]",0.538,0.785,2266,0.247


`scale_pos_weight` / SMOTE make the output **not a probability** (the reliability table shows it): any number shown to a user as "X% failure risk" would need a separate
calibration step, and calibration is only meaningful if the model has real lift — which scenario G does not.

## 5. Can engine, brakes and electrical be predicted *separately*?
Tested four ways: component labels inside Dataset 1, then Datasets 4 and 5, each with **all columns (upper bound)** and with **runtime-only** features.

In [9]:
COMP = {"Engine Overhaul": "engine", "Cooling System Repair": "engine", "Oil Change": "engine", "Brake Service": "brakes",
        "Battery Service": "electrical", "Electrical Repair": "electrical", "Tire Replacement": "other", "Transmission Service": "other"}
req = v2[v2.Maintenance_Required == 1].copy()
req["component"] = req.Maintenance_Type.map(COMP)
runtime_cols = G_cat + G_num + ["Failure_History", "Days_Since_Last_Maintenance"]
all_cols = [c for c in req.columns if c not in drop_v2 + ["Maintenance_Type", "component"]]
def cv_auc_xgb(X, y, groups=None, n_splits=5):
    X = X.copy()
    for c in X.select_dtypes(exclude="number").columns: X[c] = X[c].astype("category")
    clf = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, enable_categorical=True, tree_method="hist", n_jobs=8, random_state=0)
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=0) if groups is None else GroupKFold(n_splits)
    p = cross_val_predict(clf, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]
    return roc_auc_score(y, p), boot_auc_ci(y, p, n=100)
rows = []
for comp in ["engine", "brakes", "electrical"]:
    y = (req.component == comp).astype(int)
    a_all, ci_all = cv_auc_xgb(req[all_cols], y); a_rt, ci_rt = cv_auc_xgb(req[runtime_cols], y)
    rows.append({"component (V2, among maintenance-required vehicles)": comp, "prevalence": y.mean(), "AUC all columns": a_all, "CI all": f"[{ci_all[0]:.3f},{ci_all[1]:.3f}]",
                 "AUC runtime-only": a_rt, "CI runtime": f"[{ci_rt[0]:.3f},{ci_rt[1]:.3f}]"})
comp_v2 = pd.DataFrame(rows); display(comp_v2)

,"component (V2, among maintenance-required vehicles)",prevalence,AUC all columns,CI all,AUC runtime-only,CI runtime
0,engine,0.332,0.624,"[0.620,0.629]",0.622,"[0.618,0.626]"
1,brakes,0.167,0.700,"[0.696,0.703]",0.696,"[0.693,0.700]"
2,electrical,0.252,0.501,"[0.497,0.506]",0.495,"[0.491,0.500]"


**Where does the component signal in V2 (AUC 0.70 brakes / 0.62 engine, "runtime-only") come from?** Decomposition by feature subset (AUC, 5-fold CV, among maintenance-required vehicles):

In [10]:
subsets = {"registry only: age + make + vehicle type": ["age", "Make_and_Model", "Vehicle_Type"],
           "only Days_Since_Last_Maintenance": ["Days_Since_Last_Maintenance"],
           "only Failure_History": ["Failure_History"]}
rows = []
for name, cols in subsets.items():
    row = {"features": name}
    for comp in ["engine", "brakes", "electrical"]:
        row[comp] = cv_auc_xgb(req[cols], (req.component == comp).astype(int))[0]
    rows.append(row)
display(pd.DataFrame(rows))
display(req.groupby("component")[["Days_Since_Last_Maintenance", "Failure_History"]].mean().rename(columns=lambda c: "mean " + c))

,features,engine,brakes,electrical
0,registry only: age + make + vehicle type,0.505,0.500,0.501
1,only Days_Since_Last_Maintenance,0.625,0.701,0.498
2,only Failure_History,0.574,0.615,0.495


,mean Days_Since_Last_Maintenance,mean Failure_History
component,,
brakes,272.549,1.119
electrical,365.305,1.657
engine,410.328,1.952
other,366.043,1.680


Registry-only features give **AUC 0.50 for every component**. The whole component signal comes from `Days_Since_Last_Maintenance` (brake-type records have a mean of ~273 days, engine-type ~410):
the generator assigns an "earlier-due" maintenance type to shorter intervals — a synthetic interval rule, available only when the plate already has history in PostgreSQL. It is *not* evidence about failures.

In [11]:
# 92k: Brake_Condition / Battery_Status
yb = (d92.Brake_Condition == "Poor").astype(int)
a_rt, ci_rt = cv_auc_xgb(d92[G_cat + G_num + ["Failure_History", "Days_Since_Last_Maintenance"]], yb)
print(f"92k  Brake_Condition=='Poor' from runtime features: AUC {a_rt:.3f}  CI [{ci_rt[0]:.3f},{ci_rt[1]:.3f}]  (it is also a clause of the label rule: Poor => Maintenance_Required=1 in 100% of rows)")
print("92k  Battery_Status distribution (should be a continuous sensor reading):", d92.Battery_Status.round(0).value_counts().head(6).to_dict(), "-> ~constant 45, std", round(d92.Battery_Status.std(), 2))

92k  Brake_Condition=='Poor' from runtime features: AUC 0.501  CI [0.498,0.506]  (it is also a clause of the label rule: Poor => Maintenance_Required=1 in 100% of rows)
92k  Battery_Status distribution (should be a continuous sensor reading): {45.0: 77617, 50.0: 10373, 46.0: 1070, 47.0: 1059, 48.0: 981, 49.0: 900} -> ~constant 45, std 1.63


In [12]:
# ---- Dataset 4 (telemetry, 50 vehicles): leave-vehicle-out CV
sens4 = [c for c in d4.columns if c not in ["vehicle_id", "brand", "timestamp", "failure_date", "failure_type", "any_fault", "engine_failure_imminent", "brake_issue_imminent", "battery_issue_imminent"]]
rt4 = pd.get_dummies(d4[["brand"]]).assign(odometer=d4.odometer_reading)
g = d4.vehicle_id
def boot_by_vehicle(y, p, groups, n=300, seed=0):
    rng = np.random.default_rng(seed); vids = groups.unique(); out = []; y = np.asarray(y); p = np.asarray(p)
    for _ in range(n):
        s = rng.choice(vids, len(vids)); idx = np.concatenate([np.where(groups.values == v)[0] for v in s])
        if 0 < y[idx].sum() < len(idx): out.append(roc_auc_score(y[idx], p[idx]))
    return np.percentile(out, [2.5, 97.5]) if out else (np.nan, np.nan)
rows = []
for tgt in ["engine_failure_imminent", "brake_issue_imminent", "battery_issue_imminent", "any_fault"]:
    y = d4[tgt]
    row = {"D4 target": tgt, "positive rows": int(y.sum()), "vehicles with >=1": int(d4.loc[y == 1, "vehicle_id"].nunique())}
    for name, X in [("runtime-only (brand+odometer)", rt4), ("all sensors (upper bound)", d4[sens4])]:
        p = cross_val_predict(RandomForestClassifier(300, min_samples_leaf=3, class_weight="balanced", random_state=0, n_jobs=8), X, y, cv=GroupKFold(5), groups=g, method="predict_proba")[:, 1]
        lo, hi = boot_by_vehicle(y, p, g)
        row[f"AUC {name}"] = roc_auc_score(y, p); row[f"CI {name}"] = f"[{lo:.2f},{hi:.2f}]"
    rows.append(row)
display(pd.DataFrame(rows))
t = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0).fit(d4[sens4], d4.any_fault)
print("D4: a depth-3 tree on the sensors reproduces the label (AUC %.4f) -> labels are sensor thresholds:" % roc_auc_score(d4.any_fault, t.predict_proba(d4[sens4])[:, 1]))
print(export_text(t, feature_names=sens4))

,D4 target,positive rows,vehicles with >=1,AUC runtime-only (brand+odometer),CI runtime-only (brand+odometer),AUC all sensors (upper bound),CI all sensors (upper bound)
0,engine_failure_imminent,7,6,0.454,"[0.30,0.65]",1.000,"[1.00,1.00]"
1,brake_issue_imminent,16,13,0.442,"[0.32,0.56]",1.000,"[1.00,1.00]"
2,battery_issue_imminent,12,12,0.495,"[0.36,0.64]",0.998,"[0.99,1.00]"
3,any_fault,35,25,0.426,"[0.35,0.53]",0.998,"[0.99,1.00]"


D4: a depth-3 tree on the sensors reproduces the label (AUC 0.9999) -> labels are sensor thresholds:
|--- brake_fluid_level_psi <= 793.32
|   |--- brake_temp_c <= 128.63
|   |   |--- class: 0
|   |--- brake_temp_c >  128.63
|   |   |--- brake_pad_wear_mm <= 7.02
|   |   |   |--- class: 0
|   |   |--- brake_pad_wear_mm >  7.02
|   |   |   |--- class: 1
|--- brake_fluid_level_psi >  793.32
|   |--- battery_voltage_v <= 11.00
|   |   |--- class: 1
|   |--- battery_voltage_v >  11.00
|   |   |--- vibration_level <= 3.85
|   |   |   |--- class: 0
|   |   |--- vibration_level >  3.85
|   |   |   |--- class: 1



In [13]:
# ---- Dataset 5 (NEV faults): runtime-eligible features vs permutation null
rt5 = ["last_service_distance", "service_frequency", "driving_distance", "time_since_last_fault"]
sens5 = [c for c in d5.columns if c not in rt5 + ["fault_type", "is_fault", "repair_cost", "downtime"]]
def cv_auc_rf(X, y, seed):
    X = pd.get_dummies(X)
    p = cross_val_predict(RandomForestClassifier(150, min_samples_leaf=5, random_state=seed, n_jobs=8), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=seed), method="predict_proba")[:, 1]
    return roc_auc_score(y, p)
rows = []
for tgt_name, y in [("any fault", d5.is_fault), ("battery_issue", (d5.fault_type == "battery_issue").astype(int)),
                    ("engine_overheating", (d5.fault_type == "engine_overheating").astype(int))]:
    real = np.mean([cv_auc_rf(d5[rt5], y, s) for s in range(5)])
    null = [cv_auc_rf(d5[rt5], pd.Series(np.random.default_rng(i).permutation(y.values)), 0) for i in range(60)]
    up = np.mean([cv_auc_rf(d5[sens5 + rt5], y, s) for s in range(2)])
    rows.append({"D5 target": tgt_name, "prevalence": y.mean(), "AUC runtime-eligible": real, "permutation null 95th pct": np.percentile(null, 95),
                 "p-value": (np.sum(np.array(null) >= real) + 1) / 61, "AUC all columns (upper bound)": up})
display(pd.DataFrame(rows))

C:\Users\milag\OneDrive\Documentos\Afeka\3rd year\PROJECT_SmartCar\ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\milag\OneDrive\Documentos\Afeka\3rd year\PROJECT_SmartCar\ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\milag\OneDrive\Documentos\Afeka\3rd year\PROJECT_SmartCar\ml\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the cu

,D5 target,prevalence,AUC runtime-eligible,permutation null 95th pct,p-value,AUC all columns (upper bound)
0,any fault,0.795,0.537,0.548,0.164,0.517
1,battery_issue,0.407,0.461,0.537,0.951,0.482
2,engine_overheating,0.195,0.524,0.546,0.180,0.526


### Component-level conclusion
| Source | Component labels exist? | Learnable from **all** columns? | Learnable from **registry-only** features? | Learnable with DB history (days since last service)? |
|---|---|---|---|---|
| D1 V2 (maintenance type among required) | engine / brakes / electrical derived from `Maintenance_Type` | brakes 0.70, engine 0.62, electrical **0.50** | **no (0.50)** | brakes 0.70 / engine 0.62 — a synthetic interval rule; electrical 0.50 |
| D1 92k | `Brake_Condition`, `Battery_Status` (≈ constant) | brake = a clause of the label rule | **no (0.50)** | no (0.50) |
| D4 telemetry | yes, per component | yes (AUC ≈ 1.0) — but the labels *are* sensor thresholds | **no** (AUC 0.43–0.50, CIs include 0.5; only 7/16/12 positive rows) | n/a |
| D5 NEVs | fault types | **no (AUC 0.48–0.53, even with every sensor)** | **no** (permutation p = 0.16–0.95) | n/a |

**None of the datasets lets us estimate a per-component failure probability from information available for an arbitrary plate.** The only structure found (V2, brakes/engine) is a synthetic maintenance-interval rule that needs a PostgreSQL service history.
Datasets 4 and 5 do not add useful information either (D4: 50 vehicles, 35 positive rows, labels are sensor thresholds; D5: 1,000 rows whose labels are not predictable even from all of its sensors).
The PDF's idea of turning D4 into age/mileage-bucket priors for D1 cannot work: D4 has no age column, and its failure rates conditional on odometer/brand are indistinguishable from chance (leave-vehicle-out AUC ≈ 0.46, CI includes 0.5).

## 6. Real, official component-level information that *is* available: open safety recalls
The Ministry of Transport publishes, **per plate**, the recalls the owner has not yet performed
([dataset `hagbalat_recall`](https://data.gov.il/he/datasets/ministry_of_transport/hagbalat_recall)). It is *not* a probability and *not* ML — it is an official fact with a defect category.

In [14]:
rec = pd.read_csv(INTERIM_DIR / "israel_open_recalls.csv")
GROUP = {"מנוע ומערכותיו": "engine", "מערכת הדלק": "engine", "מערכת הפליטה": "engine", "מערכת העברת כוח": "engine",
         "בלמים": "brakes", "חשמל אליקטרוניקה ומיזוג": "electrical"}
rec["component_group"] = rec.SUG_TAKALA.map(GROUP).fillna("other (airbags, steering, body, safety systems, ...)")
print(f"{len(rec):,} open recalls | {rec.MISPAR_RECHEV.nunique():,} distinct plates | recall types: {rec.SUG_RECALL.value_counts().to_dict()}")
g = rec.groupby("component_group").agg(open_recalls=("RECALL_ID", "size"), plates=("MISPAR_RECHEV", "nunique"), distinct_recall_ids=("RECALL_ID", "nunique"))
display(g)
display(rec.SUG_TAKALA.value_counts().to_frame("open recalls"))
print("Example record (official defect description):")
display(rec[rec.component_group == "brakes"].head(2)[["MISPAR_RECHEV", "RECALL_ID", "SUG_TAKALA", "TEUR_TAKALA", "TAARICH_PTICHA"]])

125,244 open recalls | 111,273 distinct plates | recall types: {'תקלה סידרתית בטיחותית': 122170, 'קמפיין שרות טכני': 3074}


,open_recalls,plates,distinct_recall_ids
component_group,,,
brakes,2794,2780,200
electrical,5837,5773,201
engine,22710,21932,428
"other (airbags, steering, body, safety systems, ...)",93903,83457,1086


,open recalls
SUG_TAKALA,
כריות אוויר,59815
מנוע ומערכותיו,19660
אחר,15602
היגוי ומתלים,7986
מרכב ושילדה,5971
חשמל אליקטרוניקה ומיזוג,5837
בלמים,2794
מערכת הבטיחות,2671
מערכת הדלק,1217


Example record (official defect description):


,MISPAR_RECHEV,RECALL_ID,SUG_TAKALA,TEUR_TAKALA,TAARICH_PTICHA
7,1002022,9081,בלמים,שסתום מגבר בלם,2011-11-28
15,1004111,15711,בלמים,"ייתכן כי חומרים מסוימים מטבעות אטימה, לכלוך ומ...",2024-10-28


In [15]:
gov = pd.read_csv(INTERIM_DIR / "gov_registry_sample.csv")
gov["age"] = 2026 - gov.shnat_yitzur
plates_by_group = {k: set(v.MISPAR_RECHEV) for k, v in rec.groupby("component_group")}
gov["any_open_recall"] = gov.mispar_rechev.isin(set(rec.MISPAR_RECHEV))
for k, s in plates_by_group.items():
    gov[f"recall_{k.split()[0]}"] = gov.mispar_rechev.isin(s)
print(f"Random registry sample ({len(gov):,} vehicles; contiguous chunks, NOT reliable for prevalence: the recall file lists 111k plates, i.e. ~2.6% of a 4.2M registry): share with any open recall = {gov.any_open_recall.mean():.2%}")
print(gov[[c for c in gov.columns if c.startswith('recall_')]].mean().round(4).to_dict())

Random registry sample (20,000 vehicles; contiguous chunks, NOT reliable for prevalence: the recall file lists 111k plates, i.e. ~2.6% of a 4.2M registry): share with any open recall = 0.71%
{'recall_brakes': 0.0, 'recall_electrical': 0.0006, 'recall_engine': 0.0016, 'recall_other': 0.005}


## 7. What is technically defensible (proposal — nothing implemented yet)

**Do not output "% failure risk" per component from ML trained on these datasets.** The numbers would be (a) a calendar rule in disguise (V2), (b) an OR over sensors we do not have (92k),
or (c) noise (scenario G, D4, D5), and then presented with a precision they do not have. That would also violate the SRS requirement to prevent fabricated outputs.

**Proposed Stage 2 (honest, still two-stage and still uses the Stage-1 estimate):**
1. **Rule-based maintenance-due engine** (explicitly labelled *rule-based*, never as a learned probability): inputs = vehicle age, **Model-1 km estimate with its uncertainty interval**, days/km since the last
   record of each category in `maintenance_records`, fuel type (EV/hybrid skip engine-oil rules). Output per component group = `ok / due soon / overdue / unknown (no history)`.
   Interval tables must come from manufacturer/owner-manual guidance and be configurable per make/model; generic defaults are shown as generic. The 180-day rule found in Dataset 1 is a *synthetic* example of this kind of rule, not evidence.
2. **Official open-recall flags per plate and component** (section 6) — real, per-vehicle, per-component, updated daily by the Ministry.
3. **ML only where the data supports it:** log every service/failure event with its category and odometer in `maintenance_records`; once there are enough *observed* events per component
   (rule of thumb ≥ a few hundred events per component with exposure/mileage), train survival/hazard or classifier models with time-based validation and require a pre-registered minimum lift before exposing any probability.
   The UI keeps the same three gauges (Engine / Brakes / Electrical) but shows the *rule-based/recall status* until a validated model exists.
4. **Confidence gate**: the SRS "<60% confidence → rule-based fallback" cannot use predicted probability alone (with 77% prevalence a constant 0.77 is "confident"). The gate should be
   *demonstrated lift on held-out data* (e.g. AUC CI lower bound > 0.6) plus calibration, evaluated offline; otherwise always fall back.

## 8. Summary tables

In [16]:
print("Scenario G (registry only) vs rule-recovery scenarios, XGBoost scale_pos_weight, test split:")
cols = ["dataset", "scenario", "ROC-AUC", "AUC 95% CI", "PR-AUC", "lift", "Recall", "Precision", "F1"]
both = pd.concat([res_v2, res_92])
display(both[(both.model == "XGBoost (scale_pos_weight)") | (both.model.str.startswith("Baseline"))][cols])
print("Component feasibility (V2, among required):"); display(comp_v2)

Scenario G (registry only) vs rule-recovery scenarios, XGBoost scale_pos_weight, test split:


,dataset,scenario,ROC-AUC,AUC 95% CI,PR-AUC,lift,Recall,Precision,F1
0,V2 (prevalence 30%),-,0.500,-,0.300,1.000,1.000,0.300,0.462
2,V2 (prevalence 30%),G,0.505,"[0.500, 0.511]",0.304,1.013,0.909,0.300,0.451
5,V2 (prevalence 30%),H1,1.000,"[1.000, 1.000]",1.000,3.333,1.000,1.000,1.000
8,V2 (prevalence 30%),H2,0.837,"[0.833, 0.840]",0.790,2.632,0.897,0.374,0.528
11,V2 (prevalence 30%),H,1.000,"[1.000, 1.000]",1.000,3.333,1.000,1.000,1.000
14,V2 (prevalence 30%),U (ALL columns),1.000,"[1.000, 1.000]",1.000,3.333,1.000,1.000,1.000
0,92k (prevalence 77%),-,0.500,-,0.768,1.000,1.000,0.768,0.869
2,92k (prevalence 77%),G,0.508,"[0.499, 0.518]",0.774,1.007,0.966,0.769,0.856
5,92k (prevalence 77%),H1,0.502,"[0.494, 0.512]",0.772,1.006,1.000,0.768,0.869
8,92k (prevalence 77%),H2,0.765,"[0.759, 0.772]",0.928,1.209,0.913,0.791,0.848


Component feasibility (V2, among required):


,"component (V2, among maintenance-required vehicles)",prevalence,AUC all columns,CI all,AUC runtime-only,CI runtime
0,engine,0.332,0.624,"[0.620,0.629]",0.622,"[0.618,0.626]"
1,brakes,0.167,0.700,"[0.696,0.703]",0.696,"[0.693,0.700]"
2,electrical,0.252,0.501,"[0.497,0.506]",0.495,"[0.491,0.500]"
